# Drie directe MX-gewichtsmodi / A32 — vijf-epochpilot

E4M3, E2M3 en E2M1 delen één FP32-updatemaster en gebruiken elk een eigen BN-bank. Iedere modus kwantiseert **rechtstreeks uit hetzelfde actuele FP32 G(W)**, met de schaalkeuze van dat formaat. Dit experiment bevat **geen exponent- of mantissatruncatie**.

Dit notebook leest uitsluitend opgeslagen JSON/CSV: geen training, datasets, modellen of checkpoints worden geladen. De template bevat geen onderzoeksresultaten. Synthetische softwarecontroles zijn bovenaan en in iedere figuurtitel herkenbaar gemarkeerd.

**Hoofdvergelijking:** de laatste beschikbare pilotepoch en het beste checkpoint gekozen door E4M3; telkens alle drie modi van dezelfde epoch. De pilot meet geen epoch100-accuracy en bewijst geen eindconvergentie.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # In de runkopie wordt het exacte resultaatpad ingevuld.
SAVE_FIGURES = False  # True: nieuwe analysis_<tijdstip>-map; bestaande resultaten blijven behouden.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_direct_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Laatste pilotepoch en beste checkpoint gekozen door E4M3

Het beste checkpoint is het **eerste strikte maximum van de E4M3-testaccuracy**. Een gelijke score vervangt het eerdere checkpoint niet. E2M3 en E2M1 worden op precies diezelfde epoch gerapporteerd, ook wanneer hun eigen hoogste score elders ligt. Er worden geen afzonderlijke beste epochs tot één resultaat gecombineerd.

De officiële testset wordt volgens het bestaande protocol voor checkpointselectie gebruikt. De beste score is daardoor geen onafhankelijke testschatting na selectie. De laatste beschikbare epoch blijft apart zichtbaar; bij een geslaagde pilot is dat epoch5. **Epoch100 ontbreekt totdat honderd volledige epochs aantoonbaar zijn opgeslagen.** Er wordt geen ontbrekende accuracy geëxtrapoleerd.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Train- en testaccuracy per modus

Trainaccuracy is tijdens de gezamenlijke updates verzameld. Testaccuracy wordt na iedere epoch met bevroren evaluatiestaat gemeten, op dezelfde volledige testset voor alle modi. De figuren tonen uitsluitend opgeslagen epochs.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


In [ ]:
display(Markdown('## Gemeten pilotduur en afzonderlijke honderd-epochprojectie'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Directe exports per formaat en geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

Deze pilot controleert uitvoerbaarheid, numeriek gedrag en tijd van gezamenlijke directe kwantisatie. De drie uitvoermodi zijn drie rechtstreeks afgeronde representaties, elk met de eigen standaard schaalkeuze. Er is geen bewijs geleverd dat alleen de hoogste opgeslagen MX-representatie alle modi kan voeden. De exporttabel maakt zichtbaar of alleen de gedecodeerde gewichten zijn vergeleken of ook een volledige testpass is herhaald.

A32 en FP32 beschrijven de activatie- en masterdtypes. De opgeslagen CUDA-backendprecisie staat apart in de protocoltabel; cuDNN kan intern TF32 gebruiken. De emulator meet geen native MX-snelheid of energie. Deze pilot verandert de oudere dedicated, directe tweemodus- en mantissatruncatieresultaten niet.

`SAVE_FIGURES = True` schrijft uitsluitend de vijf figuren en een analysemanifest naar een **nieuwe** map. De runner kan dit voor de runkopie inschakelen. Elke uitvoering maakt een nieuwe map; JSON, CSV, checkpoints en eerdere analyses blijven ongewijzigd.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [path.name for path in run.input_files]}))
